In [ ]:
#add imports
import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print("Num GPUs Available: ", len(gpus))
    for gpu in gpus:
        print("Found GPU: ", gpu)
else:
    print("No GPU devices found.")

In [ ]:
fashion_mnist = tf.keras.datasets.fashion_mnist.load_data()
(X_train_full, y_train_full), (X_test, y_test) = fashion_mnist
X_train, y_train = X_train_full[:-5000], y_train_full[:-5000]
X_valid, y_valid = X_train_full[-5000:], y_train_full[-5000:]
X_train, X_valid, X_test = X_train / 255, X_valid / 255, X_test / 255

class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

In [ ]:
model_norm = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=[28, 28]),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(300, activation="relu",
                          kernel_initializer="he_normal"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(100, activation="relu",
                          kernel_initializer="he_normal"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(10, activation="softmax")
])

In [ ]:
model_norm.summary()

In [ ]:
[(var.name, var.trainable) for var in model_norm.layers[1].variables]

In [ ]:
model_norm.compile(loss="sparse_categorical_crossentropy", optimizer="sgd",
              metrics=["accuracy"])
history_norm = model_norm.fit(X_train, y_train, epochs=2, validation_data=(X_valid, y_valid))

In [ ]:
# extra code - clear the name counters and set the random seed
tf.keras.backend.clear_session()
tf.random.set_seed(42)

In [ ]:
model_act = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=[28, 28]),
    tf.keras.layers.Dense(300, kernel_initializer="he_normal", use_bias=False),
    # Dense layer with no activation function and no bias, so it outputs pre-activations
    tf.keras.layers.BatchNormalization(),
    # Pre-activation is normalized and then Activation as a separate layer
    tf.keras.layers.Activation("relu"),
    tf.keras.layers.Dense(100, kernel_initializer="he_normal", use_bias=False),
    # same pattern for the second hidden layer
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation("relu"),
    tf.keras.layers.Dense(10, activation="softmax")
])

In [ ]:
# extra code – just show that the model works! 😊
model_act.compile(loss="sparse_categorical_crossentropy", optimizer="sgd",
              metrics=["accuracy"])
history_act = model_act.fit(X_train, y_train, epochs=2, validation_data=(X_valid, y_valid))

In [ ]:
# compare history_act and history_norm
plt.plot(history_act.history["accuracy"], "b-", label="Separate activation training accuracy")
plt.plot(history_act.history["val_accuracy"], "b--", label="Separate activation validation accuracy")
plt.plot(history_norm.history["accuracy"], "r-", label="BatchNorm training accuracy")
plt.plot(history_norm.history["val_accuracy"], "r--", label="BatchNorm validation accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")  
plt.legend()
plt.grid(True)
#save_fig("batchnorm_activation_history_plot")
plt.show()